In [ ]:
import numpy as np
import pandas as pd

In [ ]:
train_df = pd.read_csv('Project Data/lending_club_train.csv')
test_df = pd.read_csv('Project Data/lending_club_test.csv')

In [ ]:
print("--- Train Data Head ---")
display(train_df.head())

print("\n--- Test Data Head ---")
display(test_df.head())

In [ ]:
# Converting "issue_d" to datetime format
train_df['issue_d_dt'] = pd.to_datetime(train_df['issue_d'], errors='coerce')

In [ ]:
# Filtering for 2016 data
train_2016 = train_df[train_df['issue_d_dt'].dt.year == 2016].copy()
print(f"Original training rows: {train_df.shape[0]}")
print(f"Filtered 2016 training rows: {train_2016.shape[0]}")
print(f"Months included in 2016 train set: {sorted(train_2016['issue_d_dt'].dt.month.unique())}")

In [ ]:
# Doing the same for the test data
test_df['issue_d_dt'] = pd.to_datetime(test_df['issue_d'], errors='coerce')
test_2016 = test_df[test_df['issue_d_dt'].dt.year == 2016].copy()
print(f"Original test rows: {test_df.shape[0]}")
print(f"Filtered 2016 test rows: {test_2016.shape[0]}")
print(f"Months included in 2016 test set: {sorted(test_2016['issue_d_dt'].dt.month.unique())}")

In [ ]:
# Checking for nulls
print("--- TRAIN NULL % ---")
train_nulls = (train_2016.isnull().sum() / len(train_2016) * 100).round(2)
print(train_nulls[train_nulls > 0].sort_values(ascending=False))

print("\n--- TEST NULL % ---")
test_nulls = (test_2016.isnull().sum() / len(test_2016) * 100).round(2)
print(test_nulls[test_nulls > 0].sort_values(ascending=False))

In [ ]:
# Map sub_grade (A1..G5) to an ordinal 1..35 instead of one-hot encoding it
grades = [f"{g}{n}" for g in "ABCDEFG" for n in range(1, 6)]
grade_map = {g: i + 1 for i, g in enumerate(grades)}
train_2016['sub_grade'] = train_2016['sub_grade'].str.strip().map(grade_map)
test_2016['sub_grade'] = test_2016['sub_grade'].str.strip().map(grade_map)

Alternative model

In [ ]:
# 1. Define columns to drop (IDs, dates, and data leakage columns)
cols_to_drop = ['id', 'issue_d', 'issue_d_dt', 'loan_status', 'sub_grade', 'fico_range_low']

# 2. Separate the target variable (int_rate) from the feature sets
train_y = train_2016['int_rate']
test_y = test_2016['int_rate']

train_features = train_2016.drop(columns=cols_to_drop + ['int_rate'])
test_features = test_2016.drop(columns=cols_to_drop + ['int_rate'])

# # 3. Combine temporarily to ensure One-Hot Encoding creates identical columns for both
# combined_features = pd.concat([train_features, test_features], axis=0)
# combined_encoded = pd.get_dummies(combined_features, drop_first=True)

# 3. One-Hot Encode each set separately
train_encoded = pd.get_dummies(train_features, drop_first=True, dtype=float)
test_encoded = pd.get_dummies(test_features, drop_first=True, dtype=float)

# # 4. Split them back apart into train and test sets
# train_encoded = combined_encoded.iloc[:len(train_2016)]
# test_encoded = combined_encoded.iloc[len(train_2016):]

# 4. Align test to the TRAIN columns: categories unseen in train (e.g. home_ownership_ANY)
#    are dropped, and dummies that exist only in train are added as 0
test_encoded = test_encoded.reindex(columns=train_encoded.columns, fill_value=0.0)

# Quick check on the final shapes
print("Train features shape:", train_encoded.shape)
print("Test features shape:", test_encoded.shape)

In [ ]:
# ---------------- Baseline regression tree (default settings) ----------------
from sklearn.tree import DecisionTreeRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

def report(name, model):
    """Fit on 2016 train, score on both train and 2016 test."""
    model.fit(train_encoded, train_y)
    results = {}
    for label, X, y in [("train", train_encoded, train_y), ("test", test_encoded, test_y)]:
        pred = model.predict(X)
        results[label] = {
            "R2": r2_score(y, pred),
            "RMSE": mean_squared_error(y, pred) ** 0.5,
            "MAE": mean_absolute_error(y, pred),
        }
    print(f"\n--- {name} ---")
    print(pd.DataFrame(results).T.round(4))
    return model

# 1. Reference point: always predict the training mean
report("Mean-predictor reference", DummyRegressor(strategy="mean"))

# 2. Baseline: regression tree under default settings
baseline_tree = report("Baseline regression tree (defaults)", DecisionTreeRegressor(random_state=42))
print("Leaves:", baseline_tree.get_n_leaves(), " Depth:", baseline_tree.get_depth())

# 3. Which features does the tree rely on?
importances = pd.Series(baseline_tree.feature_importances_, index=train_encoded.columns)
print("\nTop 10 feature importances:")
print(importances.sort_values(ascending=False).head(10).round(4))

In [ ]:
# ---------------- Tuning: max_leaf_nodes x min_impurity_decrease ----------------
import itertools
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold, cross_val_score

leaf_grid = [5, 10, 20, 50, 100, 500, None]          # None = no limit (default)
impurity_grid = [0.0, 0.0001, 0.001, 0.01, 0.1]       # 0.0 = default



In [ ]:
# 5-fold CV on the 2016 TRAIN data only (shuffled, so each fold mixes Jan-Jun loans)
cv = KFold(n_splits=5, shuffle=True, random_state=42)

grid_rows = []
for max_leaves, min_dec in itertools.product(leaf_grid, impurity_grid):
    tree = DecisionTreeRegressor(max_leaf_nodes=max_leaves,
                                 min_impurity_decrease=min_dec,
                                 random_state=42)
    # CV score (negative RMSE -> flip sign), computed without touching the test set
    cv_rmse = -cross_val_score(tree, train_encoded, train_y, cv=cv,
                               scoring="neg_root_mean_squared_error").mean()

    tree.fit(train_encoded, train_y)
    p_tr, p_te = tree.predict(train_encoded), tree.predict(test_encoded)
    grid_rows.append({
        "max_leaf_nodes": max_leaves if max_leaves is not None else "None",
        "min_impurity_decrease": min_dec,
        "leaves": tree.get_n_leaves(),
        "depth": tree.get_depth(),
        "train_R2": r2_score(train_y, p_tr),
        "test_R2": r2_score(test_y, p_te),
        "test_RMSE": mean_squared_error(test_y, p_te) ** 0.5,
        "test_MAE": mean_absolute_error(test_y, p_te),
        "cv_RMSE": cv_rmse,
    })

grid_results = pd.DataFrame(grid_rows)
pd.set_option("display.width", 200)
print(grid_results.round(4).to_string(index=False))



In [ ]:
# Pick the setting using CV (lowest cv_RMSE), then report its test accuracy
best = grid_results.loc[grid_results["cv_RMSE"].idxmin()]
print("\nBest setting by cross-validation:")
print(best.to_frame().T.round(4).to_string(index=False))



In [ ]:
# Plot: effect of tree size on accuracy (min_impurity_decrease = 0)
curve = grid_results[(grid_results["min_impurity_decrease"] == 0.0)
                     & (grid_results["max_leaf_nodes"] != "None")]
plt.figure(figsize=(6, 4))
plt.plot(curve["leaves"], curve["train_R2"], "o-", label="train")
plt.plot(curve["leaves"], curve["test_R2"], "s-", label="test")
plt.xscale("log"); plt.xlabel("Number of leaves"); plt.ylabel("R²")
plt.title("Accuracy vs. tree size"); plt.legend()
plt.show()

In [ ]:
# 1. Instantiate and evaluate the best regularized tree using the grid search parameters
val = best["max_leaf_nodes"]

# Convert string 'None', NaN, or None to actual Python None; otherwise cast to int
if pd.isna(val) or val == 'None' or val is None:
    max_leaves = None
else:
    max_leaves = int(val)

best_tree = report(
    "Best Regularized Regression Tree",
    DecisionTreeRegressor(
        max_leaf_nodes=max_leaves,
        min_impurity_decrease=float(best["min_impurity_decrease"]),
        random_state=42
    )
)

# 2. Generate Partial Dependence Plots to visualize business implications for top features
from sklearn.inspection import PartialDependenceDisplay
import matplotlib.pyplot as plt

features_to_plot_1 = ['installment']
features_to_plot_2 = ['term_ 60 months', 'dti', 'fico_range_high']
features_to_plot_3 = ['revol_bal', 'loan_amnt']

fig, ax = plt.subplots(figsize=(10, 4))
PartialDependenceDisplay.from_estimator(
    best_tree, 
    train_encoded, 
    features=features_to_plot_1,
    kind='average',
    ax=ax
)
plt.suptitle("Partial Dependence of Interest Rate on Key Risk Features")
plt.tight_layout()

fig, ax = plt.subplots(figsize=(10, 4))
PartialDependenceDisplay.from_estimator(
    best_tree, 
    train_encoded, 
    features=features_to_plot_2,
    kind='average',
    ax=ax
)
plt.suptitle("Partial Dependence of Interest Rate on Key Risk Features")
plt.tight_layout()

fig, ax = plt.subplots(figsize=(10, 4))
PartialDependenceDisplay.from_estimator(
    best_tree, 
    train_encoded, 
    features=features_to_plot_3,
    kind='average',
    ax=ax
)
plt.suptitle("Partial Dependence of Interest Rate on Key Risk Features")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.tree import export_text

# Print the text-based decision rules for the top levels of your tree
tree_rules = export_text(best_tree, feature_names=list(train_encoded.columns), max_depth=3)
print(tree_rules)